# Vaardigheden week 3: typen, eenheden en ontbrekende waarden

Dit notebook hoort bij het vaardighedenspoor. Het gebruikt het inkoopboek van een verzonnen winkel en een verzonnen lijst met maandelijkse prijsveranderingen, zodat de vaardigheden los blijven van het project van deze week.

Je ziet dat een waarde betekent wat het type ervan zegt, je zet getallen die als tekst zijn opgeslagen om in getallen, je brengt elke kolom terug tot één eenheid, je vermenigvuldigt procentuele veranderingen in plaats van ze op te tellen, en je vindt ontbrekende waarden in drie vermommingen. Drie opdrachten zijn voor jou, en elk ervan heeft een CHECK-cel.

**Zo werk je ermee:** voer elke cel op volgorde uit met de afspeelknop of Shift+Enter. Lees de uitvoer voordat je verdergaat.

## 1. Dezelfde tekens, verschillende betekenissen

Een computer slaat elke waarde op als een patroon van bits. Het type van een waarde is de afspraak die zegt hoe je dat patroon leest. Dezelfde tekens kunnen tekst of een getal zijn, en de uitkomst van een berekening hangt af van wat Python ziet.

In [ ]:
print('"7" + "7" geeft', "7" + "7")                 # twee stukken tekst, aan elkaar gezet
print("7 + 7 geeft", 7 + 7)                         # twee getallen, opgeteld
print('int("7") + int("7") geeft', int("7") + int("7"))   # tekst eerst omgezet in getallen
print("Het teken 7 wordt opgeslagen als code", ord("7"))

Een type heeft ook een grootte. Een kolom met kleine hele getallen kan in 8 bits worden opgeslagen, en de volgende cel drukt af welk bereik die 8 bits kunnen bevatten. Een uitkomst die niet past, begint weer van voren af aan het bereik, en er verschijnt geen foutmelding.

In [ ]:
import numpy as np

print("een kolom van 8 bits bevat", np.iinfo(np.uint8).min, "tot en met", np.iinfo(np.uint8).max)
small = np.array([200], dtype=np.uint8)
print("200 + 100 in een kolom van 8 bits:", (small + 100)[0])
print("200 + 100 in een kolom van 64 bits:", (small.astype(np.int64) + 100)[0])

De meeste datatools slaan hele getallen op in 64 bits, dus dit gebeurt zelden per ongeluk. Het gebeurt wel als een bestand of een programma een klein type kiest om ruimte te besparen. Het foute antwoord ziet er dan uit als een gewoon getal.

## 2. Laad een bestand waarin getallen als tekst binnenkomen

Spreadsheets die op Nederlands, Duits of Frans zijn ingesteld, schrijven een komma als decimaalteken en zetten een puntkomma tussen de kolommen. De cel hieronder maakt de tekst van zo'n bestand voor het inkoopboek van een verzonnen winkel, en laadt het daarna zoals je elk bestand zou laden.

De cel bewaart ook de echte waarden in een tabel met de naam `truth`, zodat de CHECK-cellen je werk kunnen testen. Een echt bestand komt nooit met zijn waarheid erbij.

In [ ]:
import io
import pandas as pd

rng = np.random.default_rng(3)
n = 40
per_kg = {"rice": 2.5, "flour": 2.0, "sugar": 2.2, "cooking oil": 6.0}   # verzonnen prijzen per kilogram
truth = pd.DataFrame({
    "date": pd.date_range("2026-03-02", periods=n, freq="D"),
    "item": rng.choice(list(per_kg), size=n),
    "bags": rng.integers(1, 9, size=n).astype(float),
    "bag_kg": rng.choice([0.5, 1.0, 2.0, 5.0, 25.0], size=n),
})
truth["unit_price"] = (truth["bag_kg"] * truth["item"].map(per_kg) * rng.uniform(0.9, 1.1, size=n)).round(2)
truth.loc[[4, 9, 15, 21, 33], "bags"] = np.nan        # vijf aankopen waarbij niemand de zakken telde

# het bestand zoals een spreadsheet met Nederlandse instellingen het zou opslaan
bags_text = truth["bags"].map(lambda b: "" if pd.isna(b) else str(int(b)))
bags_text[9] = "unknown"
bags_text[[15, 33]] = "-1"
size_text = truth["bag_kg"].map(lambda kg: f"{kg * 1000:.0f} g" if kg < 1 else f"{kg:.0f} kg")
price_text = truth["unit_price"].map(lambda p: f"{p:.2f}".replace(".", ","))
lines = ["date;item;bags;unit_price;bag_size"] + [
    f"{d:%Y-%m-%d};{i};{b};{p};{s}"
    for d, i, b, p, s in zip(truth["date"], truth["item"], bags_text, price_text, size_text)
]
file_text = "\n".join(lines)

purchases = pd.read_csv(io.StringIO(file_text), sep=";")
purchases.head(12)

## 3. Beschrijf de typen

De vier vragen uit week 1 komen nog steeds eerst. Kijk deze keer goed naar de derde: welk type heeft elke kolom? pandas geeft een kolom met tekst aan als `object`. Een kolom met getallen die pandas als tekst leest, kun je pas middelen nadat je hem hebt omgezet.

In [ ]:
print("rijen en kolommen:", purchases.shape)
print()
print(purchases.dtypes)
print()
print("waarden in de kolom bags:", sorted(purchases["bags"].dropna().unique()))
print("eerste prijzen zoals ze geschreven zijn:", purchases["unit_price"].head(3).tolist())

## 4. Lees code die een AI-assistent schreef

We vroegen een AI-assistent: *"Zet de kolom unit_price om in getallen en geef me de gemiddelde prijs per zak."* De assistent schreef de cel hieronder. Voer hem uit en beslis dan of het antwoord klopt.

In [ ]:
# Geschreven door een AI-assistent
price_ai = purchases["unit_price"].str.replace(",", "").astype(float)
print("Gemiddelde prijs per zak:", round(price_ai.mean(), 2))

De code draait zonder foutmelding, en toch is het antwoord fout. Het bestand gebruikt een komma als decimaalteken, en de code van de assistent verwijderde elke komma, alsof die duizendtallen scheidde.

De volgende cel leest de eerste rij met de hand, zet de kolom op de juiste manier om en vergelijkt de twee gemiddelden.

In [ ]:
price = purchases["unit_price"].str.replace(",", ".").astype(float)   # de komma is het decimaalteken
print("Rij 0 zoals geschreven:", purchases.loc[0, "unit_price"], "-> het getal van de AI:", price_ai[0], "-> goed gelezen:", price[0])
print("Gemiddelde prijs per zak, goed gelezen:", round(price.mean(), 2))
print("Het antwoord van de AI was te groot met een factor:", round(price_ai.mean() / price.mean(), 1))

# CHECK: elke omgezette prijs moet gelijk zijn aan de echte prijs
assert np.allclose(price, truth["unit_price"])
print("CHECK geslaagd")

Een betere oplossing vertelt pandas bij het laden welke afspraak het bestand volgt. `sep=";"` noemt het scheidingsteken tussen de kolommen, en `decimal=","` noemt het decimaalteken.

In [ ]:
again = pd.read_csv(io.StringIO(file_text), sep=";", decimal=",")
print(again.dtypes)
print("hetzelfde gemiddelde, bij het laden goed gelezen:", round(again["unit_price"].mean(), 2))

## 5. Ontbrekende waarden in drie vermommingen

Een ontbrekende waarde ziet er niet altijd uit als een lege cel. In dit bestand komt hij in drie vermommingen voor: een lege cel, een woord (`unknown`) en een codegetal (`-1`, dat de winkel opschrijft als niemand de zakken telde). De eerste stap is ze alle drie vinden.

`pd.to_numeric(..., errors="coerce")` maakt van alles wat geen getal is een ontbrekende waarde, en doet dat zonder iets te zeggen. Maak een lijst van de waarden die verdwijnen voordat je erop vertrouwt.

In [ ]:
bags_raw = purchases["bags"]
as_number = pd.to_numeric(bags_raw, errors="coerce")
print("lege cellen:", int(bags_raw.isna().sum()))
print("woorden die geen getal zijn:", sorted(bags_raw[as_number.isna() & bags_raw.notna()].unique()))
print("rijen met de code -1:", int((as_number == -1).sum()))

De code `-1` is de gevaarlijkste vermomming, want het is een getal en pandas neemt het zonder klagen mee in het gemiddelde. Vergelijk het gemiddelde met de code erin en met de code als ontbrekende waarde.

In [ ]:
bags = as_number.where(as_number != -1)          # -1 betekent dat niemand telde, dus ook dat ontbreekt
print("Gemiddeld aantal zakken per aankoop met -1 erin:", round(as_number.mean(), 2))
print("Gemiddeld aantal zakken per aankoop met -1 als ontbrekend:", round(bags.mean(), 2))
print("aankopen met een telling:", int(bags.count()), "van", len(bags))

# CHECK: de opgeschoonde aantallen moeten gelijk zijn aan de waarheid, en ontbreken waar de waarheid ontbreekt
assert (bags.isna() == truth["bags"].isna()).all()
assert np.allclose(bags.dropna(), truth["bags"].dropna())
print("CHECK geslaagd")

## 6. Houd een opschoonlogboek bij

Elke verandering die je in een tabel aanbrengt, verandert de antwoorden die de tabel geeft. Een opschoonlogboek noemt elke verandering en de reden ervoor, zodat een lezer precies kan zien hoe het ruwe bestand de tabel werd die jij gebruikte. De cel hieronder begint een schone tabel en het logboek erbij.

In [ ]:
clean = pd.DataFrame({"date": pd.to_datetime(purchases["date"]), "item": purchases["item"]})
log = []

clean["unit_price"] = price
log.append("unit_price: de komma gelezen als decimaalteken en de prijzen als getallen opgeslagen")
clean["bags"] = bags
log.append(f"bags: lege cellen, 'unknown' en de code -1 omgezet in ontbrekende waarden; {int(bags.isna().sum())} van de {len(bags)} aankopen hebben geen telling")

for line in log:
    print("-", line)

## 7. Jouw beurt: één eenheid per kolom

De kolom `bag_size` mengt twee eenheden. Sommige zakken staan in grammen, zoals `500 g`, en andere in kilogrammen, zoals `25 kg`. Een kolom met twee eenheden kun je niet optellen of vergelijken.

Vraag je AI-assistent om code die een kolom `bag_kg` aan `clean` toevoegt, met elke zakgrootte in kilogrammen, en die één regel aan `log` toevoegt. Plak de code in de cel hieronder, voer hem uit en voer daarna de CHECK-cel uit. Reken één waarde in grammen met de hand om voordat je de code vertrouwt.

In [ ]:
# jouw beurt 1: voeg clean["bag_kg"] toe (elke zakgrootte in kilogrammen) en één regel aan log

In [ ]:
# CHECK 1
assert "bag_kg" in clean, "clean heeft nog geen kolom bag_kg"
assert np.allclose(clean["bag_kg"], truth["bag_kg"]), "sommige zakgroottes staan niet in kilogrammen"
assert len(log) >= 3, "voeg één regel toe aan het opschoonlogboek"
print("CHECK 1 geslaagd: de zakgroottes lopen van", clean["bag_kg"].min(), "tot", clean["bag_kg"].max(), "kg")
for line in log:
    print("-", line)

## 8. Jouw beurt: een prijs in de juiste eenheid

Met een prijs per zak kun je een zak van 500 g niet vergelijken met een zak van 25 kg. Met een prijs per kilogram wel. Sla in `price_per_kg` de gemiddelde prijs per kilogram van elk product op, één waarde per product, berekend uit `clean`.

Je AI-assistent stelt misschien `groupby` voor. Vraag hem die regel in woorden uit te leggen voordat je hem uitvoert.

In [ ]:
# jouw beurt 2: price_per_kg = het gemiddelde van unit_price / bag_kg per product
price_per_kg = None

In [ ]:
# CHECK 2
expected = (truth["unit_price"] / truth["bag_kg"]).groupby(truth["item"]).mean()
assert price_per_kg is not None, "price_per_kg is nog leeg"
assert np.allclose(price_per_kg.sort_index(), expected.sort_index())
print("CHECK 2 geslaagd: gemiddelde prijs per kilogram")
print(price_per_kg.sort_index().round(2))

## 9. Procent, fractie en keer

Een procentuele verandering is ook een eenheid. Een stijging van 55 procent vermenigvuldigt een prijs met 1 plus 55 gedeeld door 100. Dat getal heet de groeifactor. Een prijs die 2,5 keer zo hoog wordt, is met veel meer dan 2,5 procent gestegen.

De functie hieronder zet een procentuele verandering om in de groeifactor. De laatste twee regels laten zien waarom het verschil ertoe doet.

In [ ]:
def factor(pct):
    """Een procentuele verandering als het getal waarmee een prijs wordt vermenigvuldigd."""
    return 1 + pct / 100

print("55 procent als groeifactor:", factor(55))
print("2,5 keer zo hoog, als procentuele verandering:", (2.5 - 1) * 100)
print("Twee jaar na elkaar een stijging van 20 procent:", round(factor(20) * factor(20), 2))
print("Een stijging van 50 procent en daarna een daling van 50 procent:", factor(50) * factor(-50))

## 10. Vermenigvuldig, tel niet op

De cel hieronder maakt een verzonnen lijst met maandelijkse prijsveranderingen van een pakket goederen over 24 maanden. De verandering voor maand 1 vergelijkt de prijs van het pakket met de maand voordat de lijst begint. Twee maanden zijn nooit gemeten.

Over meerdere maanden vermenigvuldigen de veranderingen zich, want de verandering van elke maand werkt op een prijs die eerdere maanden al hebben verhoogd. Wie de procenten optelt, negeert dat, dus de som valt te laag uit als de prijzen blijven stijgen.

In [ ]:
rng = np.random.default_rng(11)
change = pd.Series(np.round(rng.normal(3.5, 2.5, size=24), 1),
                   index=pd.RangeIndex(1, 25, name="month"), name="pct_change")
change.loc[[17, 18]] = np.nan            # twee maanden die niemand mat
print("ontbrekende maanden:", change[change.isna()].index.tolist())

year1 = change.loc[1:12]
print("Jaar 1, maandelijkse veranderingen opgeteld:", round(year1.sum(), 1), "procent")
print("Jaar 1, maandelijkse veranderingen vermenigvuldigd:", round((factor(year1).prod() - 1) * 100, 1), "procent")

Samenstellen betekent elke verandering om de beurt toepassen, en verder niets. De lus hieronder begint bij een prijs van 100 en past de twaalf veranderingen van jaar 1 een voor een toe. De uitkomst moet gelijk zijn aan het samengestelde cijfer.

In [ ]:
price_level = 100.0
for pct in year1:
    price_level = price_level * factor(pct)
print("Een prijs van 100 na jaar 1:", round(price_level, 2))

# CHECK: de lus en het product geven hetzelfde antwoord
assert abs(price_level - 100 * factor(year1).prod()) < 1e-9
print("CHECK geslaagd")

## 11. Een gat breekt de keten

Als één schakel in een keten van veranderingen ontbreekt, is het totaal onbekend. pandas vertelt je dat niet. `sum()` en `prod()` slaan ontbrekende waarden over, en dat komt neer op de aanname dat er in de ontbrekende maand niets veranderde. Met `skipna=False` geeft pandas in plaats daarvan een ontbrekende waarde terug.

Het kleine voorbeeld hieronder heeft vier maanden, en maand 2 ontbreekt.

In [ ]:
example = pd.Series([10.0, np.nan, 10.0, 10.0], index=[1, 2, 3, 4])
print("veranderingen:", example.tolist())
print("opgeteld, standaard in pandas:", example.sum())
print("vermenigvuldigd, standaard in pandas:", round((factor(example).prod() - 1) * 100, 1), "procent")
print("vermenigvuldigd, skipna=False:", factor(example).prod(skipna=False))
print("maand 3 en 4, zonder gat:", round((factor(example.loc[3:4]).prod() - 1) * 100, 1), "procent")

Het eerlijke antwoord heeft twee delen. Over een stuk zonder gat kun je precies zeggen hoeveel de prijzen veranderden. Over het gat heen kun je alleen zeggen wat een aanname zou opleveren, en die aanname moet je noemen. Als de prijzen in maand 2 bijvoorbeeld niet daalden, stegen ze over de vier maanden samen minstens met het samengestelde cijfer dat pandas standaard afdrukte.

## 12. Laat het gat zien in een grafiek

Een staafgrafiek van de maandelijkse veranderingen laat de twee ontbrekende maanden zien als lege plekken. Een label bij elke plek zegt dat ook, zodat geen lezer een lege plek aanziet voor een maand zonder verandering.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(change.index, change)
for m in change[change.isna()].index:
    ax.text(m, 0.3, "geen data", rotation=90, ha="center", va="bottom", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Maandelijkse prijsverandering van een verzonnen pakket (twee maanden niet gemeten)")
ax.set_xlabel("maand")
ax.set_ylabel("verandering (procent)")
plt.tight_layout()
plt.show()

## 13. Jouw beurt: het jaar met een gat

Jaar 2 loopt van maand 13 tot maand 24, en de maanden 17 en 18 ontbreken. Vraag je AI-assistent om code die twee getallen berekent, plak die hieronder en voer de CHECK uit.

- `year2_total` is de totale prijsverandering over jaar 2, in procent, zo berekend dat het antwoord ontbreekt vanwege het gat.
- `year2_after` is de prijsverandering over de maanden 19 tot en met 24, in procent. Dat stuk heeft geen gat.

Een assistent schrijft heel goed mogelijk code die voor `year2_total` een getal teruggeeft. Als de jouwe dat doet, faalt de CHECK, en die mislukking is de les. Schrijf in één zin op wat het getal van de assistent aannam over de maanden 17 en 18.

In [ ]:
# jouw beurt 3: bereken year2_total en year2_after, allebei in procent
year2_total = None
year2_after = None

In [ ]:
# CHECK 3
assert year2_after is not None, "year2_after is nog leeg"
assert year2_total is not None, "year2_total is nog leeg: het moet een ontbrekende waarde zijn, niet None"
assert np.isnan(year2_total), "year2_total verbergt de twee ontbrekende maanden; probeer prod(skipna=False)"
assert abs(year2_after - (factor(change.loc[19:24]).prod() - 1) * 100) < 1e-6
print("CHECK 3 geslaagd: jaar 2 als geheel is onbekend, en de maanden 19 tot en met 24 veranderden met", round(year2_after, 1), "procent")

## 14. Wat je deed

Je las getallen die als tekst binnenkwamen, gaf elke kolom één eenheid, vermenigvuldigde procentuele veranderingen in plaats van ze op te tellen, vond ontbrekende waarden in drie vermommingen en weigerde een totaal dat de data niet kon dragen. Elke verandering aan het inkoopboek staat in het logboek, zodat een lezer ze allemaal kan volgen.